In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

def InceptionModule(x, f1, f3r, f3, f5r, f5, proj):
    path1 = layers.Conv2D(f1, (1, 1), padding='same', activation='relu')(x)

    path2 = layers.Conv2D(f3r, (1, 1), padding='same', activation='relu')(x)
    path2 = layers.Conv2D(f3, (3, 3), padding='same', activation='relu')(path2)

    path3 = layers.Conv2D(f5r, (1, 1), padding='same', activation='relu')(x)
    path3 = layers.Conv2D(f5, (5, 5), padding='same', activation='relu')(path3)

    path4 = layers.MaxPooling2D((3, 3), strides=(1, 1), padding='same')(x)
    path4 = layers.Conv2D(proj, (1, 1), padding='same', activation='relu')(path4)

    return layers.concatenate([path1, path2, path3, path4], axis=-1)

In [2]:
def GoogleNet(input_shape=(96, 96, 3), num_classes=10):
    inputs = layers.Input(shape=input_shape)

    x = layers.Resizing(227, 227)(inputs)
    x = layers.Conv2D(64, (7,7), strides=(2,2), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((3,3), strides=(2,2), padding='same')(x)

    x = InceptionModule(x, 64, 96, 128, 16, 32, 32)
    x = InceptionModule(x, 128, 128, 192, 32, 96, 64)
    x = layers.MaxPooling2D((3,3), strides=(2,2), padding='same')(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model

In [3]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

target_size = (96, 96)
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [4]:
model = GoogleNet(input_shape=(32, 32, 3), num_classes=10)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=15, batch_size=64, validation_split=0.1)

Epoch 1/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 191s 238ms/step - accuracy: 0.1850 - loss: 2.1205 - val_accuracy: 0.3278 - val_loss: 1.8657
Epoch 2/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 162s 208ms/step - accuracy: 0.3201 - loss: 1.8506 - val_accuracy: 0.3952 - val_loss: 1.6343
Epoch 3/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 201s 207ms/step - accuracy: 0.3990 - loss: 1.6357 - val_accuracy: 0.4030 - val_loss: 1.6246
Epoch 4/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 146s 207ms/step - accuracy: 0.4500 - loss: 1.5093 - val_accuracy: 0.5046 - val_loss: 1.3654
Epoch 5/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 202s 207ms/step - accuracy: 0.4869 - loss: 1.4088 - val_accuracy: 0.5188 - val_loss: 1.3266
Epoch 6/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 202s 208ms/step - accuracy: 0.5131 - loss: 1.3486 - val_accuracy: 0.5724 - val_loss: 1.2239
Epoch 7/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 202s 208ms/step - accuracy: 0.5411 - loss: 1.2837 - val_accuracy: 0.5646 - val_loss: 1.2322
Epoch 8/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 202s 207ms/step - accuracy: 0.5532 -

In [5]:
model.evaluate(X_test, y_test)